# Alpha-Driven RFQ Simulator

This notebook demonstrates the RFQ simulator for alpha-driven liquidity provision in corporate bond markets.

**Core question:** Does spread income from patient RFQ-based trading exceed the alpha P&L sacrificed by slower convergence to the target position?

## Contents
1. Setup and Configuration
2. Single Path Demo
3. Win-Rate Calibration Validation
4. LP vs Baseline Comparison
5. Sensitivity Analysis
6. Monte Carlo Analysis

In [ ]:
# Setup path for local development
import sys
sys.path.insert(0, '../src')

# Core imports
import numpy as np
import matplotlib.pyplot as plt

# Simulator imports
from rfq_simulator import (
    SimConfig,
    run_simulation,
    run_baseline,
    compare_strategies,
    run_batch,
    run_scenario_sweep,
)
from rfq_simulator.output import (
    plot_simulation_results,
    plot_pnl_decomposition,
    create_rfq_dataframe,
    generate_summary_report,
)

# Style
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Setup and Configuration

Create a configuration with default parameters from the spec.

In [ ]:
# Create configuration with default parameters
cfg = SimConfig(
    # Simulation control
    T_days=60,
    dt_minutes=5.0,
    seed=42,
    
    # Price process
    p0=100.0,
    sigma_daily_bps=50.0,  # IG volatility
    kappa_daily=0.02,
    
    # Alpha signal
    IC=0.10,  # Moderate IC
    alpha_horizon_days=10.0,
    signal_refresh_hours=24.0,
    
    # RFQ arrivals
    rfq_rate_per_day=15.0,
    n_dealers_mean=4.0,
    
    # Position limits
    q_max=20,
    gamma=1.0,
)

cfg.validate()
print("Configuration validated successfully")
print(f"Total simulation: {cfg.T_days} days, {cfg.n_steps} price steps")
print(f"Expected RFQs: ~{cfg.rfq_rate_per_day * cfg.T_days:.0f}")

## 2. Single Path Demo

Run a single simulation path to visualize the strategy behavior.

In [ ]:
# Run single simulation
result = run_simulation(cfg, verbose=True)

In [ ]:
# Display summary
print("\n" + "="*60)
summary = result.summary()
for k, v in summary.items():
    if isinstance(v, float):
        print(f"{k}: {v:,.2f}")
    else:
        print(f"{k}: {v}")

In [ ]:
# Visualize results
fig = plot_simulation_results(result)

In [ ]:
# P&L decomposition
fig = plot_pnl_decomposition(result)

In [ ]:
# Create DataFrame for detailed analysis
df = create_rfq_dataframe(result)
if df is not None:
    print(f"RFQ log: {len(df)} events")
    display(df.head(10))
    
    # Fill statistics
    print(f"\nFill rate: {df['filled'].mean():.1%}")
    print(f"Avg spread (filled): {df[df['filled']]['spread_pnl'].mean():.2f}")

## 3. Win-Rate Calibration Validation

Verify that estimated win probabilities match realized fill rates.

In [ ]:
from rfq_simulator.output.diagnostics import plot_win_rate_calibration

fig = plot_win_rate_calibration(result, n_bins=15)

## 4. LP vs Baseline Comparison

Compare the LP (liquidity provider) strategy against the naive aggressor baseline.

In [ ]:
# Run baseline on same price path
baseline = run_baseline(
    prices=result.prices,
    regime_path=result.regime_path,
    cfg=cfg,
    seed=cfg.seed,
    verbose=True,
)

In [ ]:
# Compare strategies
comparison = compare_strategies(result, baseline)

print("\nSTRATEGY COMPARISON")
print("="*50)
for k, v in comparison.items():
    print(f"{k}: {v:,.2f}" if isinstance(v, float) else f"{k}: {v}")

print("\n" + "="*50)
if comparison['spread_minus_alpha_loss'] > 0:
    print(">>> LP STRATEGY WINS <<<")
    print(f"Net benefit: ${comparison['spread_minus_alpha_loss']:,.2f}")
else:
    print(">>> BASELINE WINS <<<")
    print(f"LP shortfall: ${-comparison['spread_minus_alpha_loss']:,.2f}")

In [ ]:
# Generate full report
report = generate_summary_report(result, baseline)
print(report)

## 5. Sensitivity Analysis

Sweep key parameters to understand strategy behavior.

In [ ]:
# Sweep IC (information coefficient)
ic_sweep = run_scenario_sweep(
    base_cfg=cfg,
    param_name='IC',
    param_values=[0.03, 0.05, 0.08, 0.10, 0.12, 0.15],
    n_paths_per_scenario=20,
    verbose=True,
)

In [ ]:
# Plot IC sensitivity
ics, pnls = ic_sweep.get_metric_vs_param('pnl_mean')
_, baseline_pnls = ic_sweep.get_metric_vs_param('baseline_pnl_mean')

plt.figure(figsize=(10, 6))
plt.plot(ics, pnls, 'b-o', label='LP Strategy', markersize=8)
plt.plot(ics, baseline_pnls, 'r--s', label='Baseline', markersize=8)
plt.xlabel('Information Coefficient (IC)')
plt.ylabel('Mean P&L ($)')
plt.title('P&L vs Information Coefficient')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Sweep RFQ rate
rfq_sweep = run_scenario_sweep(
    base_cfg=cfg,
    param_name='rfq_rate_per_day',
    param_values=[5, 10, 15, 20, 30, 40],
    n_paths_per_scenario=20,
    verbose=True,
)

In [ ]:
# Plot RFQ rate sensitivity
rates, pnls = rfq_sweep.get_metric_vs_param('pnl_mean')
_, spreads = rfq_sweep.get_metric_vs_param('spread_pnl_mean')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rates, pnls, 'b-o', markersize=8)
axes[0].set_xlabel('RFQs per Day')
axes[0].set_ylabel('Mean Total P&L ($)')
axes[0].set_title('Total P&L vs RFQ Rate')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rates, spreads, 'g-o', markersize=8)
axes[1].set_xlabel('RFQs per Day')
axes[1].set_ylabel('Mean Spread P&L ($)')
axes[1].set_title('Spread P&L vs RFQ Rate')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Monte Carlo Analysis

Run many paths to get statistical confidence in results.

In [ ]:
# Run batch simulation
batch = run_batch(
    cfg=cfg,
    n_paths=100,
    run_baseline_flag=True,
    verbose=True,
)

In [ ]:
# Display statistics
stats = batch.statistics()

print("\nMONTE CARLO STATISTICS")
print("="*50)
for k, v in stats.items():
    print(f"{k}: {v:,.2f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
# P&L distribution
pnls = batch.get_pnl_array()
baseline_pnls = np.array([r.total_pnl for r in batch.baseline_results])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(pnls, bins=20, alpha=0.7, label='LP Strategy')
axes[0].hist(baseline_pnls, bins=20, alpha=0.7, label='Baseline')
axes[0].axvline(np.mean(pnls), color='blue', linestyle='--', label=f'LP Mean: ${np.mean(pnls):,.0f}')
axes[0].axvline(np.mean(baseline_pnls), color='orange', linestyle='--', label=f'BL Mean: ${np.mean(baseline_pnls):,.0f}')
axes[0].set_xlabel('Total P&L ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('P&L Distribution')
axes[0].legend()

# Outperformance
diff = pnls - baseline_pnls
axes[1].hist(diff, bins=20, alpha=0.7, color='green')
axes[1].axvline(0, color='red', linestyle='-', linewidth=2)
axes[1].axvline(np.mean(diff), color='blue', linestyle='--', label=f'Mean: ${np.mean(diff):,.0f}')
outperf_rate = (diff > 0).mean()
axes[1].set_xlabel('LP - Baseline P&L ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'LP Outperformance (Rate: {outperf_rate:.1%})')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confidence interval for mean P&L
from scipy import stats as scipy_stats

mean_pnl = np.mean(pnls)
se_pnl = np.std(pnls) / np.sqrt(len(pnls))
ci_95 = scipy_stats.t.interval(0.95, len(pnls)-1, loc=mean_pnl, scale=se_pnl)

print(f"Mean P&L: ${mean_pnl:,.2f}")
print(f"95% CI: [${ci_95[0]:,.2f}, ${ci_95[1]:,.2f}]")
print(f"Sharpe Ratio: {stats['sharpe_of_pnls']:.2f}")

---

## Conclusion

The simulator allows us to answer the core question:

**Does spread income exceed alpha lost from slower convergence?**

Key findings:
1. P&L decomposition shows the trade-off between alpha capture and spread earning
2. Win-rate calibration validates the competitor model
3. Sensitivity analysis reveals optimal operating regions
4. Monte Carlo provides statistical confidence

Use the batch runner and scenario sweeps to explore the full parameter space.